# Baseline Category-Aggregate Feature Pipeline — From Scratch

Builds the category-aggregate feature set required by `01_baseline_forecast_model.md`
directly from the two raw source tables (`products_rows.csv`, `product_prices_rows.csv`),
through the commodity filter, Daraz-only scoping, and daily collapse, then computes the
mandatory baselines and a per-category model.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os
import sys
from pathlib import Path

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "vendor" / "pipeline" / "utils").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("vendor/pipeline/utils not found; run `git submodule update --init --recursive`")

sys.path.insert(0, str(REPO_ROOT / "vendor" / "pipeline"))

from utils.commodity_filters import is_valid_commodity, get_commodity_filter_reason

OUTDIR = "data/output/forecast_data/forecast_data_4m"
os.makedirs(OUTDIR, exist_ok=True)

pd.set_option("display.max_columns", 60)

## 1. Load products, apply commodity filter

In [ ]:
products_raw = pd.read_csv("data/raw/products_rows.csv")

products_raw["passes_filter"] = products_raw.apply(
    lambda row: is_valid_commodity(row["name"], row["category"]),
    axis=1,
)
removed = products_raw[~products_raw["passes_filter"]]
print(f"{len(removed)} / {len(products_raw)} products fail the commodity filter")
print(removed["category"].value_counts())

products = products_raw[products_raw["passes_filter"]].drop(columns=["passes_filter"]).reset_index(drop=True)

## 2. Load prices, merge onto filtered products

In [ ]:
prices = pd.read_csv("data/raw/product_prices_rows.csv")
prices = prices[prices["pipeline_version"] == "2.0.0"].reset_index(drop=True)

merged = prices.merge(
    products[["id", "name", "category", "source", "url", "keyword"]],
    left_on="product_id", right_on="id", how="left", suffixes=("", "_product")
)
orphaned = merged["id_product"].isna()
merged = merged[~orphaned].drop(columns=["id_product"]).reset_index(drop=True)

missing_unit = merged["unit_value"].isna() | merged["unit_type"].isna() | merged["unit_price"].isna()
merged = merged[~missing_unit].reset_index(drop=True)

print(f"products: {len(products)} | prices (2.0.0 only): {len(prices)}")
print(f"orphaned (no product match): {orphaned.sum()} | missing unit fields: {missing_unit.sum()}")
print(f"rows remaining: {len(merged)}")

## 3. Base prep — discount flags, calendar fields, Daraz-only

In [ ]:
df = merged[[
    "scraped_at", "run_id", "name", "category", "source", "url",
    "unit_price", "unit_type", "unit_value",
    "final_price", "original_price", "sale_price", "rating", "review_count",
]].copy()
df["scraped_at"] = pd.to_datetime(df["scraped_at"])

df["has_rating"] = df["rating"].notna()
df["has_review"] = df["review_count"].notna()
df["rating_clean"] = df["rating"].fillna(0)
df["review_count_clean"] = df["review_count"].fillna(0)

df["is_discounted"] = df["sale_price"].notna() & df["original_price"].notna() & (df["sale_price"] < df["original_price"])
df["discount_pct"] = np.nan
m = df["is_discounted"]
df.loc[m, "discount_pct"] = ((df.loc[m, "original_price"] - df.loc[m, "sale_price"]) / df.loc[m, "original_price"] * 100).round(2)

df["hour"] = df["scraped_at"].dt.hour
df["day_of_week"] = df["scraped_at"].dt.dayofweek
df["week"] = df["scraped_at"].dt.isocalendar().week.astype(int)
df["month"] = df["scraped_at"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6])

daraz_df = df[df["source"] == "daraz"].sort_values(["url", "scraped_at"]).reset_index(drop=True)
print(f"{len(daraz_df)} rows, {daraz_df['url'].nunique()} products")

## 4. Collapse to daily (modeling grain)

One row per product per day. `unit_price` takes the day's median; discount/rating
fields take the max across whichever session(s) exist that day.

In [ ]:
d = daraz_df.copy()
d["obs_date"] = d["scraped_at"].dt.date

daraz_daily = d.groupby(["url", "obs_date"]).agg(
    category=("category", "first"), source=("source", "first"), name=("name", "first"),
    unit_type=("unit_type", "first"), unit_value=("unit_value", "first"),
    unit_price=("unit_price", "median"), final_price=("final_price", "median"),
    original_price=("original_price", "median"), sale_price=("sale_price", "median"),
    is_discounted=("is_discounted", "max"), discount_pct=("discount_pct", "max"),
    rating=("rating", "max"), review_count=("review_count", "max"),
    sessions_that_day=("unit_price", "size"),
).reset_index()

daraz_daily["obs_date"] = pd.to_datetime(daraz_daily["obs_date"])
daraz_daily["day_of_week"] = daraz_daily["obs_date"].dt.dayofweek
daraz_daily["week"] = daraz_daily["obs_date"].dt.isocalendar().week.astype(int)
daraz_daily["month"] = daraz_daily["obs_date"].dt.month
daraz_daily["is_weekend"] = daraz_daily["day_of_week"].isin([5, 6])
daraz_daily["has_rating"] = daraz_daily["rating"].notna()
daraz_daily["has_review"] = daraz_daily["review_count"].notna()
daraz_daily["rating_clean"] = daraz_daily["rating"].fillna(0)
daraz_daily["review_count_clean"] = daraz_daily["review_count"].fillna(0)

print(f"{len(d)} session rows -> {len(daraz_daily)} product-days "
      f"({(daraz_daily['sessions_that_day'] == 2).sum()} had both sessions)")

df = daraz_daily.sort_values(["url", "obs_date"]).reset_index(drop=True)
df.shape

## 5. Category + unit_type scope

Only ghee, rice, sugar, tea are cleared this phase. Ghee and tea both have a small
minority of `liter` rows — split checked below and only the dominant unit_type per
category is kept, so the category-daily median isn't blended across units.

In [ ]:
CLEARED_CATEGORIES = ["rice", "ghee", "pulses"]

scoped = df[df["category"].isin(CLEARED_CATEGORIES)].copy()
print(scoped.groupby("category")["unit_type"].value_counts())

In [ ]:
dominant_unit = (
    scoped.groupby("category")["unit_type"]
    .agg(lambda s: s.value_counts().idxmax())
)
print(dominant_unit)

before = len(scoped)
scoped = scoped[
    scoped.apply(lambda r: r["unit_type"] == dominant_unit[r["category"]], axis=1)
].copy()
print(f"Dropped {before - len(scoped)} minority-unit_type rows "
      f"({(before - len(scoped)) / before:.1%})")
scoped.shape

## 6. Category-aggregate series

In [ ]:
category_daily = (
    scoped.groupby(["category", "obs_date"])["unit_price"]
    .median()
    .reset_index()
    .sort_values(["category", "obs_date"])
    .reset_index(drop=True)
)
category_daily.groupby("category")["obs_date"].agg(["min", "max", "count"])

## 7. Category-aggregate features

In [ ]:
cg = category_daily.groupby("category")["unit_price"]

category_daily["cat_price_lag_1"] = cg.shift(1)
category_daily["cat_price_lag_2"] = cg.shift(2)
category_daily["cat_price_lag_7"] = cg.shift(7)
category_daily["cat_rolling_mean_7"] = cg.transform(
    lambda s: s.rolling(7, min_periods=1).mean()
)
category_daily["cat_price_change_pct"] = cg.pct_change()

category_daily.head(10)

## 8. Discount features (category-aggregate grain)

In [ ]:
discount_daily = (
    scoped.groupby(["category", "obs_date"])
    .agg(
        is_discounted=("is_discounted", "mean"),
        discount_pct=("discount_pct", "mean"),
        has_discount=("is_discounted", "max"),
    )
    .reset_index()
)

dg = discount_daily.sort_values(["category", "obs_date"]).groupby("category")["is_discounted"]
discount_daily["rolling_discount_rate"] = dg.transform(
    lambda s: s.rolling(7, min_periods=1).mean()
)

category_daily = category_daily.merge(
    discount_daily, on=["category", "obs_date"], how="left"
)
category_daily.head(10)

## 9. Sanity check — feature coverage on the cleared categories

In [ ]:
feature_cols = [
    "cat_price_lag_1", "cat_price_lag_2", "cat_price_lag_7",
    "cat_rolling_mean_7", "cat_price_change_pct",
    "is_discounted", "discount_pct", "has_discount", "rolling_discount_rate",
]
category_daily.groupby("category")[feature_cols].apply(lambda g: g.isna().mean()).round(3)

## 10. Mandatory baselines

1. Naive persistence: `forecast(t+1) = price(t)`
2. 7-day rolling mean: `forecast(t+1) = cat_rolling_mean_7(t)`
3. Discount-aware naive: persistence, but if `is_discounted` rises at t+1
   vs t, apply the empirical median drop instead

Target: next-day `unit_price`, kept consistent across all four categories.

In [ ]:
category_daily = category_daily.sort_values(["category", "obs_date"]).reset_index(drop=True)
category_daily["target_next_price"] = category_daily.groupby("category")["unit_price"].shift(-1)

category_daily["discount_delta"] = category_daily.groupby("category")["is_discounted"].diff()

def median_drop_on_new_discount(g):
    newly_discounted = g[g["discount_delta"] > 0]
    if len(newly_discounted) == 0:
        return 0.0
    return (newly_discounted["cat_price_change_pct"].median())

median_drop_by_cat = category_daily.groupby("category").apply(median_drop_on_new_discount)
print("Empirical median pct-drop on newly-discounted days, by category:")
print(median_drop_by_cat)

In [ ]:
category_daily["baseline_naive"] = category_daily["unit_price"]
category_daily["baseline_rolling7"] = category_daily["cat_rolling_mean_7"]

def discount_aware_forecast(row):
    base = row["unit_price"]
    if row["discount_delta"] is not None and row["discount_delta"] > 0:
        drop_pct = median_drop_by_cat.get(row["category"], 0.0)
        return base * (1 + drop_pct)
    return base

category_daily["baseline_discount_aware"] = category_daily.apply(discount_aware_forecast, axis=1)

category_daily[["category", "obs_date", "unit_price", "target_next_price",
                "baseline_naive", "baseline_rolling7", "baseline_discount_aware"]].head(10)

## 11. Feature-based model — one per category (not pooled)

Categories differ enough in volatility that pooling would let noisy categories
dominate loss. Simple linear regression per category on the cleared feature set.

In [ ]:
MODEL_FEATURES = [
    "cat_price_lag_1", "cat_price_lag_2", "cat_price_lag_7",
    "cat_rolling_mean_7", "is_discounted", "discount_pct", "rolling_discount_rate",
]

results = []
models = {}

for cat in CLEARED_CATEGORIES:
    sub = category_daily[category_daily["category"] == cat].copy()
    sub[MODEL_FEATURES] = sub[MODEL_FEATURES].fillna(0)
    sub = sub.dropna(subset=["target_next_price"])

    if len(sub) < 10:
        print(f"Skipping {cat}: only {len(sub)} usable rows")
        continue

    X = sub[MODEL_FEATURES]
    y = sub["target_next_price"]

    model = LinearRegression().fit(X, y)
    models[cat] = model
    pred = model.predict(X)

    row = {"category": cat, "n": len(sub)}
    row["model_mae"] = mean_absolute_error(y, pred)
    row["model_rmse"] = mean_squared_error(y, pred) ** 0.5

    valid = sub.dropna(subset=["baseline_naive"])
    row["naive_mae"] = mean_absolute_error(valid["target_next_price"], valid["baseline_naive"])
    row["naive_rmse"] = mean_squared_error(valid["target_next_price"], valid["baseline_naive"]) ** 0.5

    valid7 = sub.dropna(subset=["baseline_rolling7"])
    row["rolling7_mae"] = mean_absolute_error(valid7["target_next_price"], valid7["baseline_rolling7"])
    row["rolling7_rmse"] = mean_squared_error(valid7["target_next_price"], valid7["baseline_rolling7"]) ** 0.5

    validd = sub.dropna(subset=["baseline_discount_aware"])
    row["discount_aware_mae"] = mean_absolute_error(validd["target_next_price"], validd["baseline_discount_aware"])
    row["discount_aware_rmse"] = mean_squared_error(validd["target_next_price"], validd["baseline_discount_aware"]) ** 0.5

    results.append(row)

results_df = pd.DataFrame(results).round(3)
results_df

## 12. Win check

A "win" is the feature model beating baseline #2 (7-day rolling mean) by a
non-trivial margin, consistently across ghee/rice/sugar/tea.

In [ ]:
results_df["model_beats_rolling7_mae"] = results_df["model_mae"] < results_df["rolling7_mae"]
results_df["mae_improvement_vs_rolling7_pct"] = (
    (results_df["rolling7_mae"] - results_df["model_mae"]) / results_df["rolling7_mae"] * 100
).round(1)
results_df[["category", "n", "model_mae", "rolling7_mae", "naive_mae",
            "discount_aware_mae", "model_beats_rolling7_mae", "mae_improvement_vs_rolling7_pct"]]

## 13. Save outputs

In [ ]:


category_daily.to_csv(f"{OUTDIR}/category_daily_baseline_features.csv", index=False)
results_df.to_csv(f"{OUTDIR}/baseline_model_results.csv", index=False)

print("Saved:")
print(f" - {OUTDIR}/category_daily_baseline_features.csv", category_daily.shape)
print(f" - {OUTDIR}/baseline_model_results.csv", results_df.shape)